# Schritt-fuer-Schritt: Residuallast-Prognose

## Worum geht es?

In dieser Arbeit versuchen wir vorherzusagen, wie viel Strom konventionelle Kraftwerke (Gas, Kohle, etc.) in Deutschland noch liefern muessen — die sogenannte **Residuallast**.

**Residuallast = Gesamtverbrauch - Windstrom - Solarstrom**

Warum ist das wichtig? Weil Deutschland immer mehr erneuerbare Energien nutzt, aber Wind und Sonne nicht immer scheinen. Netzbetreiber muessen wissen, wie viel "Luecke" in den naechsten Stunden und Tagen zu erwarten ist, um das Stromnetz stabil zu halten.

---

### Unser Ansatz

Wir vergleichen **drei verschiedene Prognose-Paradigmen**:

| Modell | Paradigma | Komplexitaet |
|--------|-----------|--------------|
| **Lineare Regression** | Klassische Statistik (unsere Baseline) | Niedrig |
| **XGBoost** (Chen & Guestrin 2016) | Gradient Boosted Trees (baumbasiertes Ensemble) | Mittel |
| **TFT** (Temporal Fusion Transformer) | Deep Learning (Transformer + LSTM) | Hoch |

Jedes Modell testen wir in **2 Strategien**:
- **Direkt**: Residuallast direkt vorhersagen
- **Indirekt**: Gesamtlast, Wind und Solar einzeln vorhersagen, dann Residuallast berechnen

Und fuer **2 Zeithorizonte**:
- **Day-Ahead (24h)**: Vorhersage fuer die naechsten 24 Stunden
- **Week-Ahead (168h)**: Vorhersage fuer die naechste Woche

---

### Aufbau dieses Notebooks

1. **Daten laden** — Stromdaten (SMARD) und Wetterdaten (Open-Meteo)
2. **Daten vorbereiten** — Zusammenfuehren, Features erstellen, aufteilen
3. **Regression trainieren** — Unsere Baseline
4. **Ergebnisse anschauen** — Vorhersagen vs. echte Werte
5. **XGBoost und TFT** — Die komplexeren Modelle (nach separatem Training)
6. **Modellvergleich** — Welches Modell ist am besten?
7. **XAI** — Welche Features sind wichtig? (Erklaerbare KI)

In [ ]:
# === Setup ===
import sys, os
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yaml
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize": (14, 6), "figure.dpi": 150,
    "font.size": 12, "font.family": "serif",
})
sns.set_palette("colorblind")

with open("config/config.yaml") as f:
    config = yaml.safe_load(f)

print("Konfiguration geladen.")
print(f"Zeitraum: {config['data']['smard']['start_date']} bis {config['data']['smard']['end_date']}")
print(f"Prognose-Horizonte: {config['models']['forecast_horizons']}")

---

## Schritt 1: Daten laden

### 1.1 SMARD — Stromdaten der Bundesnetzagentur

Die [SMARD-Plattform](https://www.smard.de) stellt oeffentliche Stromdaten fuer Deutschland bereit. Wir laden:

| Variable | Beschreibung | SMARD-Filter |
|----------|-------------|-------------|
| `residual_load` | Residuallast (unser Zielwert) | 4359 |
| `total_load` | Gesamter Stromverbrauch | 410 |
| `solar` | PV-Einspeisung | 4068 |
| `wind_onshore` | Windkraft an Land | 4067 |
| `wind_offshore` | Windkraft auf See | 1225 |

In [ ]:
# Stromdaten laden (aus Cache oder API)
smard = pd.read_csv("data/raw/smard_data.csv", index_col=0, parse_dates=True)

print(f"SMARD-Daten: {smard.shape[0]:,} Stunden x {smard.shape[1]} Spalten")
print(f"Zeitraum: {smard.index.min().date()} bis {smard.index.max().date()}")
print(f"Fehlende Werte: {smard.isna().sum().sum()}")
print(f"\nSpalten: {list(smard.columns)}")
smard.head(3)

### 1.2 Open-Meteo — Wetterdaten

Wetter beeinflusst sowohl den Stromverbrauch (Temperatur → Heizung/Kuehlung) als auch die Erzeugung (Wind → Windkraft, Sonne → Solar). Wir laden Daten von **7 Standorten** in Deutschland.

**Besonderheit:** Wir verwenden drei verschiedene Gewichtungen, weil Windraeder, Solaranlagen und Stromverbraucher an unterschiedlichen Orten stehen:

| Gewichtung | Schwerpunkt | Beispiel |
|-----------|------------|----------|
| `weather_load_*` | Bevoelkerung/Verbrauch | NRW, Muenchen |
| `weather_wind_*` | Windkraft-Kapazitaet | Hamburg, Berlin, Leipzig |
| `weather_solar_*` | Solar-Kapazitaet | Muenchen, Stuttgart |

In [ ]:
weather = pd.read_csv("data/raw/weather_data.csv", index_col=0, parse_dates=True)

print(f"Wetterdaten: {weather.shape[0]:,} Stunden x {weather.shape[1]} Spalten")
print(f"Das sind: 3 Gewichtungen x 6 Wettervariablen = 18 Spalten")
print(f"\nWettervariablen pro Gewichtung:")
load_cols = [c.replace('weather_load_', '') for c in weather.columns if c.startswith('weather_load_')]
for v in load_cols:
    print(f"  - {v}")

weather.head(3)

---

## Schritt 2: Daten vorbereiten (Preprocessing)

Bevor wir Modelle trainieren, muessen wir die Daten aufbereiten:

1. **Zusammenfuehren**: SMARD + Wetter auf gemeinsamen Zeitindex
2. **Features erstellen**: Kalenderinfos (Stunde, Wochentag, Feiertage) mit zyklischer Kodierung
3. **Fehlende Werte**: Lineare Interpolation (max. 6 Stunden Luecke)
4. **Aufteilen**: Training (70%), Validierung (15%), Test (15%) — chronologisch!
5. **Skalieren**: Werte auf 0-1 normieren (wichtig fuer Deep Learning)

**Warum chronologisch aufteilen?** Weil wir die Zukunft vorhersagen wollen. Wenn wir zufaellig aufteilen, wuerde das Modell "in die Zukunft schauen" koennen — das waere Schummelei.

In [ ]:
from src.data.preprocessing import (
    combine_data, add_time_features, handle_missing_values,
    split_data, create_darts_datasets,
)

prep_cfg = config["preprocessing"]

# 2.1 Zusammenfuehren
combined = combine_data(smard, weather, prep_cfg["target_frequency"])
print(f"Nach Zusammenfuehren: {combined.shape[0]:,} Zeilen, {combined.shape[1]} Spalten")

# 2.2 Zeitfeatures
combined = add_time_features(combined)
print(f"Nach Feature Engineering: {combined.shape[1]} Spalten")

# 2.3 Fehlende Werte
combined = handle_missing_values(combined, prep_cfg["interpolation_method"], prep_cfg["max_gap_hours"])
print(f"Fehlende Werte nach Interpolation: {combined.isna().sum().sum()}")

# 2.4 Aufteilen
train, val, test = split_data(combined, prep_cfg["train_ratio"], prep_cfg["val_ratio"])

print(f"\n=== Datensplit ===")
print(f"Training:    {len(train):>6,} Stunden ({train.index.min().date()} bis {train.index.max().date()})")
print(f"Validierung: {len(val):>6,} Stunden ({val.index.min().date()} bis {val.index.max().date()})")
print(f"Test:        {len(test):>6,} Stunden ({test.index.min().date()} bis {test.index.max().date()})")

In [ ]:
# Visualisierung des Splits
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train["residual_load"], label="Training (70%)", linewidth=0.3, color="tab:blue")
ax.plot(val.index, val["residual_load"], label="Validierung (15%)", linewidth=0.3, color="tab:orange")
ax.plot(test.index, test["residual_load"], label="Test (15%)", linewidth=0.3, color="tab:green")
ax.set_ylabel("Residuallast [MW]")
ax.set_title("Train / Validierung / Test Split")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Training: Modell lernt aus diesen Daten")
print("Validierung: Modell wird waehrend des Trainings hierauf geprueft (vermeidet Overfitting)")
print("Test: Finale Bewertung — diese Daten hat das Modell nie gesehen")

In [ ]:
# 2.5 Darts-Format erstellen
# Darts ist die Bibliothek, die wir fuer die Zeitreihenmodelle verwenden.
# Sie braucht die Daten in einem speziellen Format (TimeSeries-Objekte).

# Direkte Strategie: Residuallast als Zielwert
datasets_direct = create_darts_datasets(train, val, test, target_col="residual_load")

print("=== Darts-Datasets (direkte Strategie) ===")
print(f"Zielwert: residual_load")
print(f"Past Covariates ({len(datasets_direct['past_cov_cols'])} Spalten):")
print(f"  Energiedaten: total_load, solar, wind_onshore, wind_offshore, wind_total")
print(f"  Wetterdaten:  18 Spalten (3 Gewichtungen x 6 Variablen)")
print(f"Future Covariates ({len(datasets_direct['future_cov_cols'])} Spalten):")
print(f"  {datasets_direct['future_cov_cols']}")
print(f"\nPast Covariates = Daten aus der Vergangenheit (Modell sieht nur historische Werte)")
print(f"Future Covariates = Daten, die wir auch fuer die Zukunft kennen (Kalender, Feiertage)")

---

## Schritt 3: Regressionsbaseline trainieren

Wir starten mit dem einfachsten Modell: **Lineare Regression**.

**Was macht das Modell?**
- Es schaut sich die letzten 168 Stunden (1 Woche) der Residuallast und der Wetterdaten an
- Es lernt lineare Zusammenhaenge (z.B. "wenn es letzte Stunde windig war, ist die Residuallast jetzt wahrscheinlich niedrig")
- Es gibt pro Forecast-Schritt einen Punktwert aus — das ist unsere einfachste Baseline

**Warum "Baseline"?** Weil es das einfachste sinnvolle Modell ist. Die komplexeren Modelle (XGBoost, TFT) muessen besser sein als die Regression, sonst lohnt sich die Komplexitaet nicht.

In [ ]:
from src.models.regression_model import (
    build_regression_model, train_regression, predict_regression, get_quantile_predictions,
)
from darts.metrics import mae, rmse
from darts import TimeSeries
import time

reg_cfg = config["models"]["regression"]
quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]

# === Day-Ahead (24h) ===
print("=== Regression: Day-Ahead (24h) ===")
reg_24h = build_regression_model(
    output_chunk_length=24, lags=reg_cfg["lags"], quantiles=quantiles
)

t0 = time.time()
reg_24h = train_regression(
    reg_24h, datasets_direct["target_train_scaled"],
    past_cov_train=datasets_direct["past_cov_train"],
    future_cov_train=datasets_direct["future_cov_train"],
)
print(f"Trainingszeit: {time.time()-t0:.0f} Sekunden")

# Vorhersage
pred_24h_scaled = predict_regression(
    reg_24h, n=24, series=datasets_direct["target_val_scaled"],
    past_covariates=datasets_direct["past_cov_full"],
    future_covariates=datasets_direct["future_cov_full"], num_samples=1,
)
pred_24h = datasets_direct["target_scaler"].inverse_transform(pred_24h_scaled)
actual_24h = datasets_direct["target_test"][:24]

mae_24 = mae(actual_24h, pred_24h[:len(actual_24h)])
rmse_24 = rmse(actual_24h, pred_24h[:len(actual_24h)])
print(f"\nErgebnis (24h):")
print(f"  MAE  = {mae_24:,.0f} MW (durchschnittlicher Fehler)")
print(f"  RMSE = {rmse_24:,.0f} MW (bestraft grosse Ausreisser staerker)")

In [ ]:
# === Week-Ahead (168h) ===
print("=== Regression: Week-Ahead (168h) ===")
reg_168h = build_regression_model(
    output_chunk_length=168, lags=reg_cfg["lags"], quantiles=quantiles
)

t0 = time.time()
reg_168h = train_regression(
    reg_168h, datasets_direct["target_train_scaled"],
    past_cov_train=datasets_direct["past_cov_train"],
    future_cov_train=datasets_direct["future_cov_train"],
)
print(f"Trainingszeit: {time.time()-t0:.0f} Sekunden")

pred_168h_scaled = predict_regression(
    reg_168h, n=168, series=datasets_direct["target_val_scaled"],
    past_covariates=datasets_direct["past_cov_full"],
    future_covariates=datasets_direct["future_cov_full"], num_samples=1,
)
pred_168h = datasets_direct["target_scaler"].inverse_transform(pred_168h_scaled)
actual_168h = datasets_direct["target_test"][:168]

mae_168 = mae(actual_168h, pred_168h[:len(actual_168h)])
rmse_168 = rmse(actual_168h, pred_168h[:len(actual_168h)])
print(f"\nErgebnis (168h):")
print(f"  MAE  = {mae_168:,.0f} MW")
print(f"  RMSE = {rmse_168:,.0f} MW")

---

## Schritt 4: Ergebnisse visualisieren

### 4.1 Vorhersage vs. echte Werte

In [ ]:
# Day-Ahead Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 24h
a24 = actual_24h.pd_dataframe()
p24 = pred_24h[:len(actual_24h)].pd_dataframe()
axes[0].plot(a24.index, a24.values, 'o-', label='Ist-Werte', color='black', linewidth=2, markersize=4)
axes[0].plot(p24.index, p24.values, 's-', label='Vorhersage (Regression)', color='tab:blue', linewidth=2, markersize=4)
axes[0].set_ylabel('Residuallast [MW]')
axes[0].set_title(f'Day-Ahead (24h) — MAE = {mae_24:,.0f} MW')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# 168h
a168 = actual_168h.pd_dataframe()
p168 = pred_168h[:len(actual_168h)].pd_dataframe()
axes[1].plot(a168.index, a168.values, label='Ist-Werte', color='black', linewidth=1.5)
axes[1].plot(p168.index, p168.values, label='Vorhersage (Regression)', color='tab:blue', linewidth=1.5)
axes[1].set_ylabel('Residuallast [MW]')
axes[1].set_title(f'Week-Ahead (168h) — MAE = {mae_168:,.0f} MW')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%a %d.%m'))

plt.tight_layout()
plt.show()

print("Schwarze Linie = echte Werte (was tatsaechlich passiert ist)")
print("Blaue Linie = was das Modell vorhergesagt hat")
print("Je naeher die Linien beieinander, desto besser das Modell.")

### 4.2 Fehleranalyse

Wo und wann macht das Modell die groessten Fehler?

In [ ]:
# Residuen (Fehler) analysieren
residuals = actual_168h.values().flatten() - pred_168h[:len(actual_168h)].values().flatten()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Fehler ueber die Zeit
colors = ['tab:red' if r > 0 else 'tab:blue' for r in residuals]
axes[0].bar(range(len(residuals)), residuals, color=colors, alpha=0.7, width=1)
axes[0].axhline(y=0, color='black', linewidth=0.8)
axes[0].set_title('Fehler ueber die Zeit (168h)')
axes[0].set_xlabel('Stunde')
axes[0].set_ylabel('Fehler [MW]')
axes[0].text(0.02, 0.95, 'rot = unterschaetzt\nblau = ueberschaetzt',
             transform=axes[0].transAxes, fontsize=9, va='top')

# 2) Fehlerverteilung
axes[1].hist(residuals, bins=30, edgecolor='black', linewidth=0.5, color='steelblue')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[1].axvline(x=np.mean(residuals), color='orange', linestyle='--',
                label=f'Mittel: {np.mean(residuals):,.0f} MW')
axes[1].set_title('Fehlerverteilung')
axes[1].set_xlabel('Fehler [MW]')
axes[1].legend()

# 3) Ist vs Vorhersage Scatter
a_vals = actual_168h.values().flatten()
p_vals = pred_168h[:len(actual_168h)].values().flatten()
axes[2].scatter(a_vals, p_vals, alpha=0.6, s=30)
lims = [min(a_vals.min(), p_vals.min()), max(a_vals.max(), p_vals.max())]
axes[2].plot(lims, lims, 'r--', linewidth=1.5, label='Perfekte Vorhersage')
axes[2].set_xlabel('Ist-Werte [MW]')
axes[2].set_ylabel('Vorhersage [MW]')
axes[2].set_title('Ist vs. Vorhersage')
axes[2].legend()

plt.suptitle('Regression Baseline — Fehleranalyse', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Mittlerer Fehler: {np.mean(residuals):,.0f} MW (ideal: 0)")
print(f"Std. Abweichung:  {np.std(residuals):,.0f} MW")
print(f"Max Fehler:       {np.max(np.abs(residuals)):,.0f} MW")

---

## Schritt 5: XGBoost und TFT

Die beiden komplexeren Modelle werden separat trainiert, weil XGBoost und TFT deutlich mehr Rechenzeit brauchen. XGBoost trainiert typischerweise in **3-8 Minuten** (120 interne Modelle: 5 Quantile x 24 Horizont-Schritte), TFT je nach Hardware **30-60 Minuten**.

**Zum Trainieren:** Fuehre im Terminal aus:
```bash
python main.py run --model all --target residual_load --horizon day_ahead
```

### Was macht XGBoost anders als die Regression?

**XGBoost** (Chen & Guestrin 2016, KDD) ist ein **Gradient-Boosted-Trees-Ensemble**:

1. **Baumbasiert**: Es lernt verschachtelte if/else-Regeln aus den Daten (z.B. "wenn Stunde >= 18 UND Temperatur < 5°C → hohe Last erwartet")
2. **Boosting**: Es baut hunderte flacher Baeume nacheinander — jeder neue Baum korrigiert die Fehler seiner Vorgaenger
3. **Nichtlinear**: Im Gegensatz zur Regression erkennt es komplexe Interaktionen zwischen Features
4. **Probabilistisch**: Native Quantile-Regression (`objective="reg:quantileerror"` ab XGBoost 2.0) liefert dieselben 5 Quantile wie TFT
5. **Direkte Mehrschritt-Strategie**: `multi_models=True` — fuer jeden Horizont-Schritt (1 h, 2 h, …, 24 h) wird ein eigenes Modell trainiert. Keine Fehler-Akkumulation, identisch zur Regression-Baseline.
6. **Stabil auf Apple Silicon**: `tree_method="hist"` mit eigener Thread-Infrastruktur — wir haben XGBoost statt LightGBM gewaehlt, weil LightGBMs OpenMP-Runtime auf macOS in Kombination mit `multi_models=True` reproduzierbar SIGSEGV ausloest.

XGBoost ist in vielen Kaggle- und M-Competitions der Standard fuer tabulare Zeitreihen und gewinnt regelmaessig gegen neuronale Modelle bei moderaten Datenmengen.

### Was macht der TFT anders?

Der **Temporal Fusion Transformer** (Lim et al. 2021) kombiniert mehrere Ideen aus dem Deep Learning:

1. **Variable Selection Network**: Lernt automatisch, welche Features wichtig sind
2. **Attention-Mechanismus**: Kann gezielt auf bestimmte Zeitpunkte in der Vergangenheit "achten"
3. **LSTM-Schichten**: Versteht lang- und kurzfristige zeitliche Muster
4. **Probabilistisch**: Native Quantile-Regression wie XGBoost
5. **Erklaerbar**: Attention Weights und Variable-Importance sind direkt abfragbar (XAI!)

### Paradigmenvergleich

Alle drei Modelle haben **dieselbe Feature-Basis** (1 Woche Lookback, 17 Kovariaten). XGBoost und TFT prognostizieren dieselben 5 Quantile. Unterschiedlich ist nur das Lern-Paradigma — genau das vergleichen wir:

| Modell | Lernt | Nichtlinearitaet | Mehrschritt-Strategie |
|--------|-------|------------------|-----------------------|
| Regression | Gewichtete Summe der Features | Keine | Direkt (`multi_models=True`) |
| XGBoost | Hunderte Entscheidungsbaeume | via Tree-Splits | Direkt (`multi_models=True`) |
| TFT | Neuronales Netz mit Attention | via Aktivierungsfunktionen | Joint Encoder-Decoder |

Regression und XGBoost unterscheiden sich damit **ausschliesslich im Paradigma** (linear vs. tree-based Ensemble) — nicht zusaetzlich in der Mehrschritt-Strategie. Das ist wissenschaftlich der sauberste Vergleich.

In [ ]:
# Ergebnisse laden (falls vorhanden)
from pathlib import Path

results_file = Path("output/results/comparison_table.csv")
if results_file.exists():
    comparison = pd.read_csv(results_file)
    print("=== Modellvergleich (alle Modelle) ===")
    display(comparison)
else:
    print("Noch keine vollstaendigen Ergebnisse vorhanden.")
    print("Fuehre 'python main.py' aus, um alle Modelle zu trainieren.")
    print(f"\nBisherige Regression-Ergebnisse:")
    print(f"  24h:  MAE = {mae_24:,.0f} MW, RMSE = {rmse_24:,.0f} MW")
    print(f"  168h: MAE = {mae_168:,.0f} MW, RMSE = {rmse_168:,.0f} MW")

---

## Schritt 6: Metriken erklaert

### Punktvorhersage-Metriken

| Metrik | Beschreibung | Einheit |
|--------|-------------|--------|
| **MAE** | Mittlerer absoluter Fehler — "Im Durchschnitt liegt die Vorhersage X MW daneben" | MW |
| **RMSE** | Wurzel des mittleren quadratischen Fehlers — wie MAE, aber bestraft grosse Fehler staerker | MW |
| **MAPE** | Prozentualer Fehler | % |

### Probabilistische Metriken

| Metrik | Beschreibung |
|--------|-------------|
| **Pinball Loss** | Bewertet, wie gut die Quantile kalibriert sind (niedriger = besser) |
| **Coverage 80%** | Anteil der echten Werte, die im 80%-Konfidenzintervall liegen (ideal: 80%) |
| **Interval Width** | Breite des Konfidenzintervalls (schmaler = praeziser, aber nur wenn Coverage stimmt) |

### Diebold-Mariano Test

Der DM-Test sagt uns, ob der Unterschied zwischen zwei Modellen **statistisch signifikant** ist oder nur Zufall:
- p < 0.05: Ja, ein Modell ist wirklich besser
- p >= 0.05: Der Unterschied koennte Zufall sein

---

## Schritt 7: XAI (Erklaerbare KI)

Ein grosser Vorteil des TFT ist seine **Erklaerbarkeit**. Im Gegensatz zu vielen Black-Box-Modellen kann der TFT uns zeigen:

1. **Welche Variablen sind am wichtigsten?** (Variable Selection Weights)
2. **Auf welche Zeitpunkte achtet das Modell?** (Attention Weights)

Fuer die Regression verwenden wir **SHAP-Werte** — eine Methode, die fuer jede einzelne Vorhersage erklaert, welche Features wie stark beigetragen haben.

Diese Analyse wird nach dem Training der Deep-Learning-Modelle durchgefuehrt.

```python
# Beispiel-Code fuer TFT Explainability:
from src.explainability.xai import explain_tft, plot_tft_variable_selection
xai = explain_tft(model=tft_model, ...)
plot_tft_variable_selection(xai["explainer"])
```

---

## Zusammenfassung und naechste Schritte

### Was wir bisher gemacht haben:
1. Stromdaten von SMARD und Wetterdaten von Open-Meteo geladen (2021-2025)
2. Drei verschiedene Wetter-Gewichtungen erstellt (Last, Wind, Solar)
3. Daten in Train/Val/Test aufgeteilt (70/15/15)
4. Regressionsbaseline trainiert und evaluiert

### Was noch kommt:
- XGBoost und TFT Training (Tree-Ensemble + Deep Learning)
- Indirekte Strategie (Gesamtlast, Wind, Solar einzeln vorhersagen)
- Vollstaendiger Modellvergleich mit allen Metriken
- XAI-Analyse (Feature Importance bei XGBoost, Attention Weights bei TFT)

### Ergebnismatrix (Ziel):

| Modell | Strategie | 24h MAE | 24h RMSE | 168h MAE | 168h RMSE |
|--------|-----------|---------|----------|----------|----------|
| Regression | Direkt | ... | ... | ... | ... |
| Regression | Indirekt | ... | ... | ... | ... |
| XGBoost | Direkt | ... | ... | ... | ... |
| XGBoost | Indirekt | ... | ... | ... | ... |
| TFT | Direkt | ... | ... | ... | ... |
| TFT | Indirekt | ... | ... | ... | ... |

### So fuehrst du die vollstaendige Pipeline aus:
```bash
cd /path/to/Code
source .venv/bin/activate
python main.py run --model all --target residual_load --horizon day_ahead
```